# VirtualWorks Internship — Task 2: Customer Churn Analysis

Fictional laptop-repair business. Goal: identify behavioral patterns associated with customer churn and build a leakage-safe classification model.

## Business definition
A customer is considered **churned** when the `Churn` target equals 1. The analysis focuses on service inactivity, support contacts, satisfaction, repair frequency, tenure, and customer-plan characteristics. Churn rate is the share of customers lost in the analysis population. citeturn0search0

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report

df = pd.read_csv('customer_churn_laptop_repair.csv')
df.head()

In [ ]:
print('Shape:', df.shape)
print('Churn rate:', df['Churn'].mean())
print(df.isna().sum())

## Leakage prevention
The target `Churn` and identifier `Customer_ID` are excluded from the predictors. Preprocessing (imputation, scaling and one-hot encoding) is fitted inside a scikit-learn Pipeline using only the training split. This prevents test-set information from influencing training.

In [ ]:
X = df.drop(columns=['Churn','Customer_ID'])
y = df['Churn']

num_cols = ['Tenure_Months','Repairs_Last_12M','Support_Contacts','Days_Since_Last_Service',
            'Satisfaction_Score','Avg_Response_Hours','Annual_Spend']
cat_cols = ['City','Device_Brand','Plan','Warranty_Active','Discount_Used']

preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

model = Pipeline([
    ('prep', preprocess),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced'))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:,1]

print('Accuracy:', accuracy_score(y_test,pred))
print('Precision:', precision_score(y_test,pred,zero_division=0))
print('Recall:', recall_score(y_test,pred,zero_division=0))
print('ROC-AUC:', roc_auc_score(y_test,proba))
print(confusion_matrix(y_test,pred))

## Interpretation
For retention work, precision and recall should be considered together. Recall answers how many actual churners the model catches; precision indicates how often a customer flagged as high risk really churns. This matters when deciding how broadly to target retention campaigns.

In [ ]:
print(df.groupby('Plan')['Churn'].mean().sort_values(ascending=False))
print(df.groupby('Warranty_Active')['Churn'].mean().sort_values(ascending=False))
print(df.groupby(pd.cut(df['Days_Since_Last_Service'], [0,30,60,90,180], labels=['0-30','31-60','61-90','91-180']))['Churn'].mean())

## Recommended retention actions
1. Contact customers showing long service inactivity before they disengage completely.
2. Prioritize low-satisfaction customers for proactive support follow-up.
3. Review response-time performance for customers with repeated support contacts.
4. Offer targeted maintenance reminders or service benefits to high-risk customers rather than sending the same campaign to everyone.
5. Re-evaluate the value proposition of plans/segments with above-average churn.

These recommendations should be validated with future customer outcomes before being treated as causal conclusions.